# Домашняя работа: квантизация и дистилляция LLM

В этой работе вы самостоятельно проведёте эксперименты с квантизацией и дистилляцией LLM и проверите, сколько качества при этом теряет модель. Задача, на которую обучаемся, — разбор запросов к голосовому ассистенту из датасета MASSIVE (ru-RU): по запросу модель возвращает JSON с intent и slots. Подробнее: https://huggingface.co/datasets/AmazonScience/massive

### Сперва настроим окружение и подготовим код для экспериментов

In [14]:
# Раскомментируйте в новом окружении, затем перезапустите kernel:
# %pip install "torch==2.8.0" "transformers==4.55.2" "datasets==4.1.1" \
#     "accelerate==1.10.1" "llmcompressor==0.8.1" "compressed-tensors==0.12.2" \
#     "numpy==2.2.6" "pandas==2.2.3" "safetensors>=0.5" "huggingface-hub<1" tqdm
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "7")
os.environ.setdefault("HF_HOME", os.path.abspath(".cache/huggingface"))


'/home/saumnov/study/HSE_LLM/7/.cache/huggingface'

In [2]:
import gc
import json
import math
import random
import re
import shutil
import struct
from collections import Counter
from collections.abc import Sequence
from contextlib import nullcontext
from dataclasses import asdict, dataclass
from importlib.metadata import version
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import Dataset, DatasetDict, load_dataset
from IPython.display import display
from tqdm.auto import tqdm
from transformers import (
    AutoConfig, AutoModelForCausalLM, AutoTokenizer,
    CompressedTensorsConfig, GenerationConfig, PreTrainedModel, PreTrainedTokenizerBase,
    Trainer, TrainingArguments, set_seed,
)

/home/saumnov/study/HSE_LLM/7/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
SEED = 42
ARTIFACTS = Path("artifacts_compression_hw")
ARTIFACTS.mkdir(exist_ok=True)
set_seed(SEED)

def clean_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def ensure_empty_dir(path: str | Path) -> Path:
    path = Path(path)
    if path.exists() and any(path.iterdir()):
        raise FileExistsError(f"{path} уже существует и не пустой")
    path.mkdir(parents=True, exist_ok=True)
    return path

environment = {
    p: version(p) for p in (
        "torch", "transformers", "datasets", "llmcompressor",
        "compressed-tensors", "numpy", "safetensors",
    )
}
environment.update(cuda=torch.version.cuda, gpu=torch.cuda.get_device_name(0))
print(environment)
(ARTIFACTS / "environment.json").write_text(
    json.dumps(environment, indent=2), encoding="utf-8"
)

{'torch': '2.8.0+cu126', 'transformers': '4.55.2', 'datasets': '4.1.1', 'llmcompressor': '0.8.1', 'compressed-tensors': '0.12.2', 'numpy': '2.2.6', 'safetensors': '0.6.2', 'cuda': '12.6', 'gpu': 'NVIDIA H100 80GB HBM3'}


239

## Загрузка MASSIVE

Используем датасет MASSIVE. Все калибровочные примеры и данные для дистилляции берём только из train. Оцениваем на валидации (dev).

In [4]:
DATASET_ID = "AmazonScience/massive"
DATASET_REVISION = "ed58ac423a2f4121720918bf5301577edce4ffd3"  # refs/convert/parquet
LOCALE = "ru-RU"

dataset_url = f"hf://datasets/{DATASET_ID}@{DATASET_REVISION}/{LOCALE}"
data_files = {
    "train": f"{dataset_url}/train/0000.parquet",
    "validation": f"{dataset_url}/validation/0000.parquet",
    "test": f"{dataset_url}/test/0000.parquet",
}
raw = load_dataset("parquet", data_files=data_files)
print(raw)

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 11514 examples [00:00, 453431.88 examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating validation split: 2033 examples [00:00, 237567.77 examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating test split: 2974 examples [00:00, 351544.69 examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'locale', 'partition', 'scenario', 'intent', 'utt', 'annot_utt', 'worker_id', 'slot_method', 'judgments'],
        num_rows: 11514
    })
    validation: Dataset({
        features: ['id', 'locale', 'partition', 'scenario', 'intent', 'utt', 'annot_utt', 'worker_id', 'slot_method', 'judgments'],
        num_rows: 2033
    })
    test: Dataset({
        features: ['id', 'locale', 'partition', 'scenario', 'intent', 'utt', 'annot_utt', 'worker_id', 'slot_method', 'judgments'],
        num_rows: 2974
    })
})


In [5]:
example_index = 0
example_raw = raw["train"][example_index]
print("Исходная запись MASSIVE:")
print(json.dumps(
    {key: example_raw[key] for key in ("id", "utt", "annot_utt", "intent")},
    ensure_ascii=False, indent=2,
))

Исходная запись MASSIVE:
{
  "id": "1",
  "utt": "разбуди меня в девять утра в пятницу",
  "annot_utt": "разбуди меня в [time : девять утра] в [date : пятницу]",
  "intent": 48
}


### Сконвертируем данные для калибровки

В MASSIVE текст запроса лежит в поле `utt`, intent записан числом, а slots отмечены прямо в тексте поля `annot_utt`, например `[time : семь утра]`. Модель должна по запросу вернуть JSON с названием intent и списком slots в виде пар «тип — значение». Поэтому преобразуем запись в `text` — исходный запрос — и `gold` — эталонный JSON-ответ. Такой формат удобнее для калибровки и подсчёта метрик.

In [6]:
SLOT_PATTERN = re.compile(r"\[([^:\[\]]+?)\s*:\s*([^\[\]]*?)\]")

def extract_slots(annot_utt: str) -> list[list[str]]:
    """Достаёт slots из разметки MASSIVE вида `[time : семь утра]`.

    Args:
        annot_utt: размеченный запрос из поля `annot_utt`.

    Returns:
        Список пар `[тип, значение]` в порядке появления в запросе.
    """
    return [[m.group(1).strip(), m.group(2).strip()]
            for m in SLOT_PATTERN.finditer(annot_utt)]

def target_json(intent: str, slots: list[list[str]]) -> str:
    """Собирает эталонный ответ в компактный JSON.

    Args:
        intent: название intent.
        slots: список пар `[тип, значение]`.

    Returns:
        Строка `{"intent":...,"slots":[...]}` без пробелов между полями.
    """
    return json.dumps(
        {"intent": intent, "slots": slots},
        ensure_ascii=False, separators=(",", ":"),
    )

def prepare_massive(raw_splits: DatasetDict) -> DatasetDict:
    """Переводит сплиты MASSIVE в формат запрос - эталонный JSON.

    Args:
        raw_splits: исходные сплиты MASSIVE с полями `id`, `utt`, `intent`, `annot_utt`.

    Returns:
        Сплиты с полями `id`, `text` (запрос) и `gold` (эталонный JSON).
    """
    intent_feature = raw_splits["train"].features["intent"]

    def convert(row: dict[str, Any]) -> dict[str, str]:
        intent = intent_feature.int2str(row["intent"])
        slots = extract_slots(row["annot_utt"])
        return {
            "id": str(row["id"]), "text": row["utt"],
            "gold": target_json(intent, slots),
        }

    return DatasetDict({
        split: ds.map(convert, remove_columns=ds.column_names)
        for split, ds in raw_splits.items()
    })

In [20]:

data = prepare_massive(raw)
INTENTS = list(raw["train"].features["intent"].names)
# game_type входит в общую схему MASSIVE, но отсутствует в ru-RU train.
SLOT_TYPES = sorted({
    kind for row in raw["train"]
    for kind, value in extract_slots(row["annot_utt"])
} | {"game_type"})
train_data = data["train"]
DEV_SAMPLES = None
dev_data = data["validation"]
if DEV_SAMPLES is not None:
    dev_data = dev_data.shuffle(seed=SEED).select(range(min(DEV_SAMPLES, len(dev_data))))
print("Число классов:", len(INTENTS), "типов slots:", len(SLOT_TYPES))

Число классов: 60 типов slots: 55


In [8]:
example_prepared = train_data[example_index]
print("Запрос модели (text):")
print(example_prepared["text"])
print("\nЭталонный ответ (gold):")
print(json.dumps(json.loads(example_prepared["gold"]), ensure_ascii=False, indent=2))

Запрос модели (text):
разбуди меня в девять утра в пятницу

Эталонный ответ (gold):
{
  "intent": "alarm_set",
  "slots": [
    [
      "time",
      "девять утра"
    ],
    [
      "date",
      "пятницу"
    ]
  ]
}


### Генерация ответов
Всем моделям даём одинаковую инструкцию и одинаковые описания допустимых классов. Генерация детерминирована: greedy decoding, одинаковый лимит токенов.

In [26]:
BASE_MODEL_ID = "Qwen/Qwen3-8B" # можно пробовать модели того же семейства, но поменьше, чтобы считалось быстрее

BASE_MODEL_REVISION = "b968826d9c46dd6066d109eabc6255188de91218"

# Значения используются как пример, при необходимости можно регулировать
MAX_PROMPT_TOKENS = 2048
MAX_NEW_TOKENS = 128
GENERATE_BATCH_SIZE = 32

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cuda:0


In [21]:
system_prompt = (
    'Определи intent и извлеки slots. Верни только JSON вида '
    '{"intent":"...","slots":[["тип","значение"],...]}. '
    'Значения slots копируй из запроса. Если slots нет, верни пустой список. '
    "Не добавляй объяснения.\nДопустимые intents: " + ", ".join(INTENTS)
    + "\nДопустимые типы slots: " + ", ".join(SLOT_TYPES)
)

def encode_prompt(tokenizer: PreTrainedTokenizerBase, text: str) -> list[int]:
    """Токенизирует запрос в chat-формате с общим system prompt.

    Args:
        tokenizer: токенизатор модели с chat template.
        text: запрос пользователя.

    Returns:
        Token ids промпта, заканчивающиеся началом ответа ассистента.
    """
    ids = tokenizer.apply_chat_template(
        [{"role": "system", "content": system_prompt},
         {"role": "user", "content": text}],
        tokenize=True, add_generation_prompt=True, enable_thinking=False,
    )
    if len(ids) > MAX_PROMPT_TOKENS:
        raise ValueError(f"Prompt имеет {len(ids)} токенов. Увеличьте лимит, не обрезайте запрос.")
    return ids

def generation_config(tokenizer: PreTrainedTokenizerBase) -> GenerationConfig:
    pad_token_id = tokenizer.pad_token_id
    if pad_token_id is None:
        pad_token_id = tokenizer.eos_token_id
    return GenerationConfig(
        do_sample=False, num_beams=1, max_new_tokens=MAX_NEW_TOKENS,
        eos_token_id=tokenizer.eos_token_id, pad_token_id=pad_token_id,
        use_cache=True,
    )

def trim_answer(token_ids: list[int], eos_token_id: int, pad_token_id: int) -> list[int]:
    """Отрезает паддинг, после генерации

    Returns:
        Токены ответа; eos остаётся последним, если модель его сгенерировала.
    """
    if eos_token_id in token_ids:
        return token_ids[:token_ids.index(eos_token_id) + 1]
    while token_ids and token_ids[-1] == pad_token_id:
        token_ids.pop()
    return token_ids

@torch.inference_mode()
def generate_answer_ids(
    model: PreTrainedModel,
    tokenizer: PreTrainedTokenizerBase,
    prompts: Sequence[list[int]],
) -> list[list[int]]:
    """Жадно генерирует ответы на батч промптов.

    Args:
        model: causal LM.
        tokenizer: токенизатор модели.
        prompts: token ids промптов; паддинг добавляется слева.

    Returns:
        Токены ответа для каждого промпта в том же порядке, без промпта и паддинга.
    """
    config = generation_config(tokenizer)
    width = max(len(ids) for ids in prompts)
    input_ids, attention_mask = [], []

    for ids in prompts:
        pad = width - len(ids)
        input_ids.append([config.pad_token_id] * pad + ids)
        attention_mask.append([0] * pad + [1] * len(ids))
    input_ids = torch.tensor(input_ids, device=model.device)
    attention_mask = torch.tensor(attention_mask, device=model.device)

    output = model.generate(
        input_ids=input_ids, attention_mask=attention_mask,
        generation_config=config,
    )
    return [
        trim_answer(row, config.eos_token_id, config.pad_token_id)
        for row in output[:, input_ids.shape[1]:].tolist()
    ]

In [46]:
BASELINE_PATH = ARTIFACTS / "baseline"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, revision=BASE_MODEL_REVISION)
baseline_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID, revision=BASE_MODEL_REVISION, torch_dtype=torch.bfloat16,
    device_map=DEVICE,
).eval()
if not BASELINE_PATH.exists():
    ensure_empty_dir(BASELINE_PATH)
    baseline_model.save_pretrained(BASELINE_PATH, safe_serialization=True)
    tokenizer.save_pretrained(BASELINE_PATH)
print("Исходная модель:", BASE_MODEL_ID)
print("Пример prompt, токенов:", len(encode_prompt(tokenizer, train_data[0]["text"])))

Исходная модель: Qwen/Qwen3-8B
Пример prompt, токенов: 507


## Метрика качества

$$
Q = \tfrac{1}{2} \operatorname{IntentAccuracy} + \tfrac{1}{2} \operatorname{SlotMicroF1}.
$$

Slots сравниваются как **мультимножество** пар `(тип, значение)`: порядок не влияет, лишние дубликаты дают false positives. Неверный JSON или нарушение схемы считается ошибкой всего ответа.

Дополнительно считаем `frame_exact_match` и `valid_json_rate`. Если во всей выборке нет ни одного gold/predicted slot, slot F1 равен 1 только при валидности всех ответов, иначе 0.

In [35]:
def normalize_value(value: str) -> str:
    """Приводит значение slot к виду для сравнения."""
    return " ".join(value.casefold().split())

def parse_answer(
    text: str, intents: Sequence[str], slot_types: Sequence[str],
) -> dict[str, Any] | None:
    """Разбирает ответ модели и проверяет формат.

    Args:
        text: сырой ответ модели.
        intents: допустимые названия intent.
        slot_types: допустимые типы slots.

    Returns:
        Словарь с ключами `intent` и `slots` или None, если ответ невалиден.
    """
    try:
        obj = json.loads(text.strip())
        if not isinstance(obj, dict) or set(obj) != {"intent", "slots"}:
            return None
        if not isinstance(obj["intent"], str) or obj["intent"] not in intents:
            return None
        if not isinstance(obj["slots"], list):
            return None
        for slot in obj["slots"]:
            if not isinstance(slot, list) or len(slot) != 2:
                return None
            kind, value = slot
            if (not isinstance(kind, str) or kind not in slot_types
                    or not isinstance(value, str) or not value.strip()):
                return None
        return obj
    except (ValueError, TypeError):
        return None

def slot_counter(obj: dict[str, Any]) -> Counter:
    """Превращает slots ответа в мультимножество

    Args:
        obj: разобранный ответ с ключом `slots`.

    Returns:
        Counter пар `(тип, нормализованное значение)`.
    """
    return Counter((kind, normalize_value(value)) for kind, value in obj["slots"])

def compute_metrics(
    predictions: Sequence[str],
    golds: Sequence[str],
    intents: Sequence[str],
    slot_types: Sequence[str],
) -> dict[str, float]:
    """Считает метрики качества по ответам модели.

    Args:
        predictions: ответы модели.
        golds: эталонные JSON в том же порядке.
        intents: допустимые названия intent.
        slot_types: допустимые типы slots.

    Returns:
        Словарь с `quality`, `intent_accuracy`, `slot_micro_f1`,
        `frame_exact_match`, `valid_json_rate` и числом примеров `n`.
    """
    if len(predictions) != len(golds) or not golds:
        raise ValueError("Нужны непустые списки одинаковой длины.")
    correct_intent = exact = valid = tp = fp = fn = 0
    for text, gold_text in zip(predictions, golds):
        gold = parse_answer(gold_text, intents, slot_types)
        if gold is None:
            raise ValueError("Невалидная gold-разметка")
        pred = parse_answer(text, intents, slot_types)
        gold_slots = slot_counter(gold)
        pred_slots = Counter() if pred is None else slot_counter(pred)
        tp += sum((pred_slots & gold_slots).values())
        fp += sum((pred_slots - gold_slots).values())
        fn += sum((gold_slots - pred_slots).values())
        if pred is not None:
            valid += 1
            intent_ok = pred["intent"] == gold["intent"]
            correct_intent += int(intent_ok)
            exact += int(intent_ok and pred_slots == gold_slots)
    n = len(golds)
    denom = 2 * tp + fp + fn
    slot_f1 = 2 * tp / denom if denom else float(valid == n)
    intent_acc = correct_intent / n
    return {
        "quality": 0.5 * (intent_acc + slot_f1),
        "intent_accuracy": intent_acc,
        "slot_micro_f1": slot_f1,
        "frame_exact_match": exact / n,
        "valid_json_rate": valid / n,
        "n": n,
    }

def evaluate(
    model: PreTrainedModel, tokenizer: PreTrainedTokenizerBase, ds: Dataset,
) -> tuple[dict[str, float], list[str]]:
    """Генерирует ответы на выборку батчами и считает метрики.

    Args:
        model: оцениваемая модель.
        tokenizer: её токенизатор.
        ds: выборка с полями `text` и `gold`.

    Returns:
        Метрики из compute_metrics и список ответов модели в порядке выборки.
    """
    model.eval()
    texts = ds["text"]
    predictions = []

    for start in tqdm(range(0, len(texts), GENERATE_BATCH_SIZE)):
        prompts = [encode_prompt(tokenizer, text)
                   for text in texts[start:start + GENERATE_BATCH_SIZE]]
        answers = generate_answer_ids(model, tokenizer, prompts)
        predictions.extend(tokenizer.batch_decode(answers, skip_special_tokens=True))

    metrics = compute_metrics(predictions, ds["gold"], INTENTS, SLOT_TYPES)
    return metrics, predictions

def evaluate_checkpoint(
    model_path: str | Path, ds: Dataset,
) -> tuple[dict[str, float], list[str]]:
    """Загружает сохранённый checkpoint, оценивает его и выгружает из памяти.

    Args:
        model_path: каталог dense или квантизированного checkpoint.
        ds: выборка с полями `text` и `gold`.

    Returns:
        Метрики и ответы модели, как в evaluate.
    """
    model_path = str(model_path)
    config = AutoConfig.from_pretrained(model_path)
    quantization_kwargs = {}
    unpack_int8 = False
    if getattr(config, "quantization_config", None):
        unpack_int8 = config.quantization_config["format"] == "int-quantized"
        # Веса распаковываются в bf16; квантизация активаций сохраняется.
        quantization_kwargs["quantization_config"] = CompressedTensorsConfig(
            run_compressed=unpack_int8
        )
    model = AutoModelForCausalLM.from_pretrained(
        model_path, torch_dtype=torch.bfloat16,
        device_map=DEVICE,
        **quantization_kwargs,
    )
    if unpack_int8:
        model.hf_quantizer.compressor.decompress_model(model)
    # Веса уже лежат на сетке квантизации; повторно округлять их на каждом токене не нужно.
    for layer in model.modules():
        scheme = getattr(layer, "quantization_scheme", None)
        if scheme is not None:
            scheme.weights = None
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        return evaluate(model, tokenizer, ds)
    finally:
        del model
        clean_memory()

def save_evaluation(name, metrics, predictions, rows):
    path = ARTIFACTS / "evaluations"
    path.mkdir(exist_ok=True)
    (path / f"{name}.json").write_text(json.dumps(metrics, indent=2))
    records = [dict(id=row["id"], text=row["text"], gold=row["gold"], prediction=pred)
               for row, pred in zip(rows, predictions)]
    (path / f"{name}.jsonl").write_text(
        "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in records),
        encoding="utf-8",
    )


Посчитаем метрики бейзлайна.

In [27]:
baseline_metrics, baseline_predictions = evaluate(baseline_model, tokenizer, dev_data)
print(baseline_metrics)
save_evaluation("baseline", baseline_metrics, baseline_predictions, dev_data)
(ARTIFACTS / "baseline_dev.json").write_text(
    json.dumps(baseline_metrics, indent=2), encoding="utf-8"
)
results = [{
    "run": "baseline", "path": str(BASELINE_PATH),
    "method": "dense", "calibration": None, **baseline_metrics,
}]
del baseline_model
clean_memory()

  0%|          | 0/64 [00:00<?, ?it/s]

  2%|▏         | 1/64 [00:01<02:03,  1.97s/it]

  3%|▎         | 2/64 [00:04<02:12,  2.14s/it]

  5%|▍         | 3/64 [00:07<02:29,  2.45s/it]

  6%|▋         | 4/64 [00:10<02:41,  2.69s/it]

  8%|▊         | 5/64 [00:12<02:36,  2.65s/it]

... (57 outputs left out)

 98%|█████████▊| 63/64 [02:18<00:02,  2.29s/it]

100%|██████████| 64/64 [02:19<00:00,  2.05s/it]

100%|██████████| 64/64 [02:19<00:00,  2.19s/it]

{'quality': 0.28996661819832475, 'intent_accuracy': 0.36645351696999506, 'slot_micro_f1': 0.21347971942665447, 'frame_exact_match': 0.09591736350221347, 'valid_json_rate': 0.5258239055582883, 'n': 2033}


## Квантизация (5 баллов)

Используя код с семинара выполните следующие задания:
1) На примере одного из методов откалибруйте бейзлайн на MASSIVE train и на WikiText, сравните качество на dev. Есть ли влияние калибровочного датасета на метрику? (1 балл)

2) Поэкспериментируйте с разными конфигурациями методов AWQ, GPTQ, SmoothQuant (3 балла)

    Запустите минимум три конфигурации, пробуйте менять разные факторы:

    - битность
    - число калибровочных примеров;
    - group size
    - symmetric / asymmetric quantization;
    - smoothing_strength
    - Дополнительно (+1 балл) — какие слои оставить в исходной точности

3) Объясните результат — какие параметры наиболее значимы? (1 балл)


In [36]:
from llmcompressor import oneshot, reset_session
from llmcompressor.modifiers.awq import AWQModifier
from llmcompressor.modifiers.quantization import GPTQModifier
from llmcompressor.modifiers.smoothquant import SmoothQuantModifier
from compressed_tensors.quantization import QuantizationArgs, QuantizationScheme

(ARTIFACTS / "protocol.json").write_text(json.dumps({
    "seed": SEED, "dataset_revision": DATASET_REVISION,
    "intents": INTENTS, "slot_types": SLOT_TYPES,
    "model": BASE_MODEL_ID, "model_revision": BASE_MODEL_REVISION,
    "dev_ids": list(dev_data["id"]), "system_prompt": system_prompt,
    "generation": generation_config(tokenizer).to_dict(),
    "batch_size": GENERATE_BATCH_SIZE,
}, ensure_ascii=False, indent=2), encoding="utf-8")

calibration_rows = train_data.shuffle(seed=SEED).select(range(512))
wiki = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train",
                    revision="b08601e04326c79dfdd32d625aee71d232d685c3")
wiki = wiki.filter(lambda row: len(row["text"].split()) >= 40).shuffle(seed=SEED)
calibration = {"massive": [], "wikitext": []}
for row, wiki_row in zip(calibration_rows, wiki):
    calibration["massive"].append(encode_prompt(tokenizer, row["text"]))
    length = len(tokenizer.encode(row["text"], add_special_tokens=False))
    wiki_ids = tokenizer.encode(wiki_row["text"], add_special_tokens=False)[:length]
    calibration["wikitext"].append(encode_prompt(tokenizer, tokenizer.decode(wiki_ids)))
(ARTIFACTS / "calibration.json").write_text(json.dumps({
    "seed": SEED, "massive_ids": list(calibration_rows["id"]), "input_ids": calibration,
}))

quant_configs = [
    dict(run="gptq_massive", method="GPTQ", calibration="massive", n_calib=128, group_size=128),
    dict(run="gptq_wikitext", method="GPTQ", calibration="wikitext", n_calib=128, group_size=128),
    dict(run="awq_massive", method="AWQ", calibration="massive", n_calib=128, group_size=128),
    dict(run="smoothquant_05", method="SmoothQuant", calibration="massive", n_calib=128, alpha=0.5),
    dict(run="gptq_512", method="GPTQ", calibration="massive", n_calib=512, group_size=128),
    dict(run="gptq_g64", method="GPTQ", calibration="massive", n_calib=128, group_size=64),
    dict(run="smoothquant_08", method="SmoothQuant", calibration="massive", n_calib=128, alpha=0.8),
]

def quantize(config):
    set_seed(SEED)
    model = AutoModelForCausalLM.from_pretrained(
        BASELINE_PATH, torch_dtype=torch.bfloat16, device_map=DEVICE,
    ).eval()
    ignore = ["lm_head"]
    if config["method"] == "SmoothQuant":
        recipe = [
            SmoothQuantModifier(smoothing_strength=config["alpha"]),
            GPTQModifier(scheme="W8A8", targets="Linear", ignore=ignore),
        ]
    else:
        scheme = QuantizationScheme(targets=["Linear"], weights=QuantizationArgs(
            num_bits=4, type="int", symmetric=True,
            strategy="group", group_size=config["group_size"],
        ))
        modifier = AWQModifier if config["method"] == "AWQ" else GPTQModifier
        recipe = [modifier(config_groups={"group_0": scheme}, ignore=ignore)]
    ids = calibration[config["calibration"]][:config["n_calib"]]
    ds = Dataset.from_dict({"input_ids": ids, "attention_mask": [[1] * len(x) for x in ids]})
    oneshot(model=model, dataset=ds, recipe=recipe, tokenizer=tokenizer,
            max_seq_length=MAX_PROMPT_TOKENS, num_calibration_samples=len(ds),
            shuffle_calibration_samples=False, log_dir=str(ARTIFACTS / "quantization_logs"))
    path = ensure_empty_dir(ARTIFACTS / "quantized" / config["run"])
    model.save_pretrained(path, save_compressed=True)
    tokenizer.save_pretrained(path)
    (path / "experiment.json").write_text(json.dumps({
        **config, "seed": SEED, "calibration_tokens": sum(map(len, ids)),
    }, indent=2))
    reset_session()
    del model, recipe
    clean_memory()
    return path

results = [row for row in results if row["run"] == "baseline"]
for config in quant_configs:
    print(config)
    path = ARTIFACTS / "quantized" / config["run"]
    if not (path / "experiment.json").exists():
        quantize(config)
    metrics_path = ARTIFACTS / "evaluations" / f"{config['run']}.json"
    if metrics_path.exists():
        metrics = json.loads(metrics_path.read_text())
    else:
        metrics, predictions = evaluate_checkpoint(path, dev_data)
        save_evaluation(config["run"], metrics, predictions, dev_data)
    size = sum(p.stat().st_size for p in path.glob("*.safetensors")) / 2**30
    results.append({**config, "path": str(path), "size_gib": size, **metrics})
    pd.DataFrame(results).to_csv(ARTIFACTS / "quantization_results.csv", index=False)
    display(pd.DataFrame([results[-1]]))

display(pd.DataFrame(results).drop(columns="path"))


{'run': 'gptq_massive', 'method': 'GPTQ', 'calibration': 'massive', 'n_calib': 128, 'group_size': 128}


,run,method,calibration,n_calib,group_size,path,size_gib,quality,intent_accuracy,slot_micro_f1,frame_exact_match,valid_json_rate,n
0,gptq_massive,GPTQ,massive,128,128,artifacts_compression_hw/quantized/gptq_massive,5.654482,0.380905,0.531727,0.230084,0.10182,0.767831,2033


{'run': 'gptq_wikitext', 'method': 'GPTQ', 'calibration': 'wikitext', 'n_calib': 128, 'group_size': 128}


,run,method,calibration,n_calib,group_size,path,size_gib,quality,intent_accuracy,slot_micro_f1,frame_exact_match,valid_json_rate,n
0,gptq_wikitext,GPTQ,wikitext,128,128,artifacts_compression_hw/quantized/gptq_wikitext,5.654482,0.286237,0.368913,0.203561,0.086572,0.534186,2033


{'run': 'awq_massive', 'method': 'AWQ', 'calibration': 'massive', 'n_calib': 128, 'group_size': 128}


,run,method,calibration,n_calib,group_size,path,size_gib,quality,intent_accuracy,slot_micro_f1,frame_exact_match,valid_json_rate,n
0,awq_massive,AWQ,massive,128,128,artifacts_compression_hw/quantized/awq_massive,5.654482,0.391502,0.520905,0.262099,0.098377,0.772258,2033


... (8,126 outputs left out)

100%|██████████| 64/64 [06:48<00:00,  7.22s/it]

100%|██████████| 64/64 [06:48<00:00,  6.39s/it]

,run,method,calibration,n_calib,alpha,path,size_gib,quality,intent_accuracy,slot_micro_f1,frame_exact_match,valid_json_rate,n
0,smoothquant_08,SmoothQuant,massive,128,0.8,artifacts_compression_hw/quantized/smoothquant_08,8.790363,0.311862,0.415642,0.208082,0.11215,0.578947,2033


,run,method,calibration,quality,intent_accuracy,slot_micro_f1,frame_exact_match,valid_json_rate,n,n_calib,group_size,size_gib,alpha
0,baseline,dense,None,0.289967,0.366454,0.213480,0.095917,0.525824,2033,NaN,NaN,NaN,NaN
1,gptq_massive,GPTQ,massive,0.380905,0.531727,0.230084,0.101820,0.767831,2033,128.0,128.0,5.654482,NaN
2,gptq_wikitext,GPTQ,wikitext,0.286237,0.368913,0.203561,0.086572,0.534186,2033,128.0,128.0,5.654482,NaN
3,awq_massive,AWQ,massive,0.391502,0.520905,0.262099,0.098377,0.772258,2033,128.0,128.0,5.654482,NaN
4,smoothquant_05,SmoothQuant,massive,0.341696,0.455976,0.227415,0.124939,0.654697,2033,128.0,NaN,8.790363,0.5
5,gptq_512,GPTQ,massive,0.212005,0.282833,0.141176,0.076734,0.408756,2033,512.0,128.0,5.654482,NaN
6,gptq_g64,GPTQ,massive,0.250155,0.325627,0.174684,0.073783,0.468765,2033,128.0,64.0,5.755557,NaN
7,smoothquant_08,SmoothQuant,massive,0.311862,0.415642,0.208082,0.112150,0.578947,2033,128.0,NaN,8.790363,0.8


### Выводы

Для GPTQ калибровка на MASSIVE дала Quality 0.3809, на WikiText — 0.2862. Инструкция и длины примеров были сопоставимы, но корпуса различались и языком, и содержанием.

Переход от 128 к 512 примерам изменил Quality на -0.1689; уменьшение group size со 128 до 64 — на -0.1307. У SmoothQuant увеличение smoothing_strength с 0.5 до 0.8 дало -0.0298. Среди этих пар сильнее всего повлияло число калибровочных примеров. Большая выборка и меньшая группа здесь не улучшили результат.

Лучший вариант — AWQ: Quality 0.3915 против 0.2900 у бейзлайна. Valid JSON Rate вырос с 52.58% до 77.23%. По Frame Exact Match лидирует SmoothQuant с α=0.5 (0.1249).


## Hard-label дистилляция (5 баллов)

Теперь реализуем hard-label дистилляцию знаний от учителя **Qwen3-32B** в бейзлайн.

### Задания
1) Измерьте качество учителя на dev и сравните с бейзлайном: насколько учитель сильнее? (1 балл)
2) Получите жадные ответы учителя на сэмпле из train. Используйте батч для ускорения при необходимости (1 балл)
3) Реализуйте код для обучения ученика на ответах учителя (1 балл)
4) Проведите эксперименты с обучением, получите модель ученика, отклоняющуюся на dev не более чем на 10% от метрики учителя (2 балла)


### Измерим качество учителя и сравним с бейзлайном (1 балл)

In [37]:
STUDENT_MODEL_ID = BASE_MODEL_ID
TEACHER_MODEL_ID = "Qwen/Qwen3-32B"

TEACHER_MODEL_REVISION = "9216db5781bf21249d130ec9da846c4624c16137"

teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL_ID, revision=TEACHER_MODEL_REVISION)
if (
    tokenizer.get_vocab() != teacher_tokenizer.get_vocab()
    or tokenizer.chat_template != teacher_tokenizer.chat_template
    or tokenizer.eos_token_id != teacher_tokenizer.eos_token_id
    or tokenizer.pad_token_id != teacher_tokenizer.pad_token_id
):
    raise ValueError("Токенизаторы ученика и учителя различаются.")
del teacher_tokenizer

In [38]:
teacher = AutoModelForCausalLM.from_pretrained(
    TEACHER_MODEL_ID, revision=TEACHER_MODEL_REVISION, torch_dtype=torch.bfloat16,
    device_map=DEVICE,
).eval()
student_vocab_size = AutoConfig.from_pretrained(STUDENT_MODEL_ID, revision=BASE_MODEL_REVISION).vocab_size
if teacher.config.vocab_size != student_vocab_size:
    raise ValueError("Размерности выходных логитов teacher и student различаются.")

In [39]:
teacher_metrics, teacher_predictions = evaluate(teacher, tokenizer, dev_data)
save_evaluation("teacher", teacher_metrics, teacher_predictions, dev_data)
display(pd.DataFrame([baseline_metrics, teacher_metrics], index=["baseline", "teacher"]))
print("Прирост Quality учителя:", teacher_metrics["quality"] - baseline_metrics["quality"])


  0%|          | 0/64 [00:00<?, ?it/s]

  2%|▏         | 1/64 [00:05<06:07,  5.83s/it]

  3%|▎         | 2/64 [00:10<05:34,  5.39s/it]

  5%|▍         | 3/64 [00:17<05:55,  5.84s/it]

  6%|▋         | 4/64 [00:25<06:52,  6.88s/it]

  8%|▊         | 5/64 [00:31<06:25,  6.53s/it]

... (58 outputs left out)

100%|██████████| 64/64 [06:36<00:00,  5.48s/it]

100%|██████████| 64/64 [06:36<00:00,  6.20s/it]

,quality,intent_accuracy,slot_micro_f1,frame_exact_match,valid_json_rate,n
baseline,0.289967,0.366454,0.213480,0.095917,0.525824,2033
teacher,0.561089,0.757009,0.365169,0.281358,0.972946,2033


Прирост Quality учителя: 0.2711223243617928


### Получим ответы учителя на трейне (1 балл)

Не забывайте выгружать модели и очищать память.

In [40]:
DISTILL_SAMPLES = 4096

def collect_teacher_answers(
    model: PreTrainedModel,
    tokenizer: PreTrainedTokenizerBase,
    rows: Dataset,
    output_path: str | Path,
) -> Path:
    """Генерируем ответы учителя батчами и сохраняем их в answers.jsonl.

    Args:
        model: модель-учитель.
        tokenizer: общий токенизатор учителя и ученика.
        rows: примеры train с полями `id`, `text`, `gold`.
        output_path: пустой каталог для ответов.

    Returns:
        Каталог с answers.jsonl; в каждой строке `id`, `text`, `gold`,
        `teacher_answer`, `prompt_ids`, `answer_ids`.
    """
    output_path = ensure_empty_dir(output_path)
    records = []
    row_list = list(rows)

    for start in tqdm(range(0, len(row_list), GENERATE_BATCH_SIZE)):
        batch = row_list[start:start + GENERATE_BATCH_SIZE]
        prompts = [encode_prompt(tokenizer, row["text"]) for row in batch]
        answers = generate_answer_ids(model, tokenizer, prompts)
        texts = tokenizer.batch_decode(answers, skip_special_tokens=True)
        for row, prompt, answer, text in zip(batch, prompts, answers, texts):
            records.append({**row, "teacher_answer": text,
                            "prompt_ids": prompt, "answer_ids": answer})

    if not records:
        raise ValueError("Учитель не вернул ни одного ответа.")
    (output_path / "answers.jsonl").write_text(
        "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in records),
        encoding="utf-8",
    )
    (output_path / "config.json").write_text(json.dumps({
        "model": TEACHER_MODEL_ID, "revision": TEACHER_MODEL_REVISION,
        "seed": SEED, "n": len(records), "generation": generation_config(tokenizer).to_dict(),
    }, indent=2))
    print(f"Сохранено ответов: {len(records)} / {len(row_list)}")
    return output_path

In [41]:
distill_rows = train_data.shuffle(seed=SEED).select(
    range(min(DISTILL_SAMPLES, len(train_data)))
)
TEACHER_ANSWERS = collect_teacher_answers(
    teacher, tokenizer, distill_rows, ARTIFACTS / "teacher_answers"
)
del teacher
clean_memory()

  0%|          | 0/128 [00:00<?, ?it/s]

  1%|          | 1/128 [00:05<11:40,  5.52s/it]

  2%|▏         | 2/128 [00:11<12:02,  5.73s/it]

  2%|▏         | 3/128 [00:18<13:31,  6.49s/it]

  3%|▎         | 4/128 [00:25<13:36,  6.58s/it]

  4%|▍         | 5/128 [00:32<13:31,  6.60s/it]

... (121 outputs left out)

 99%|█████████▉| 127/128 [13:48<00:06,  6.88s/it]

100%|██████████| 128/128 [13:55<00:00,  6.99s/it]

100%|██████████| 128/128 [13:55<00:00,  6.53s/it]

Сохранено ответов: 4096 / 4096


### Реализация пайплайна обучения ученика (1 балл)

Полное дообучение Qwen3-8B на A100 80GB проходит на грани по памяти. Для решения можно включать gradient checkpointing, уменьшать batch или брать меньшую модель того же семейства.

In [42]:
class TeacherAnswerDataset(torch.utils.data.Dataset):
    """Примеры для обучения ученика на ответах учителя."""

    def __init__(self, path: str | Path) -> None:
        """

        Args:
            path: каталог с answers.jsonl из collect_teacher_answers.
        """
        self.rows = [json.loads(line) for line in
                     (Path(path) / "answers.jsonl").read_text(encoding="utf-8").splitlines()]
        self.rows = [row for row in self.rows
                     if parse_answer(row["teacher_answer"], INTENTS, SLOT_TYPES) is not None
                     and row["answer_ids"][-1] == tokenizer.eos_token_id]

    def __len__(self) -> int:
        """
            Число отобранных ответов учителя.
        """
        return len(self.rows)

    def __getitem__(self, index: int) -> dict[str, list[int]]:
        """
        Args:
            index: номер примера.

        Returns:
            Словарь с `input_ids` (промпт + ответ), `attention_mask` и `labels`,
            токены промпта маскируем значением -100.
        """
        row = self.rows[index]
        input_ids = row["prompt_ids"] + row["answer_ids"]
        labels = [-100] * len(row["prompt_ids"]) + row["answer_ids"]
        return {
            "input_ids": input_ids,
            "attention_mask": [1] * len(input_ids),
            "labels": labels,
        }

def collate_teacher_answers(features: list[dict[str, list[int]]]) -> dict[str, torch.Tensor]:
    """Паддим примеры справа до общей длины батча.

    Args:
        features: примеры из TeacherAnswerDataset.

    Returns:
        Тензоры `input_ids`, `attention_mask` и `labels` формы [batch, length];
        паддинг в `labels` равен -100.
    """
    pad_id = tokenizer.pad_token_id
    if pad_id is None:
        pad_id = tokenizer.eos_token_id
    width = max(len(feature["input_ids"]) for feature in features)
    batch = {"input_ids": [], "attention_mask": [], "labels": []}
    for feature in features:
        pad = width - len(feature["input_ids"])
        batch["input_ids"].append(feature["input_ids"] + [pad_id] * pad)
        batch["attention_mask"].append(feature["attention_mask"] + [0] * pad)
        batch["labels"].append(feature["labels"] + [-100] * pad)
    return {key: torch.tensor(value) for key, value in batch.items()}

In [43]:
training_data = TeacherAnswerDataset(TEACHER_ANSWERS)
print(f"Для обучения оставлено {len(training_data)} из {DISTILL_SAMPLES} ответов")

def make_trainer(name, n_samples, learning_rate, epochs):
    set_seed(SEED)
    student = AutoModelForCausalLM.from_pretrained(
        BASELINE_PATH, torch_dtype=torch.bfloat16,
    )
    student.config.use_cache = False
    args = TrainingArguments(
        output_dir=str(ARTIFACTS / "training_logs" / name),
        num_train_epochs=epochs, learning_rate=learning_rate,
        per_device_train_batch_size=4, gradient_accumulation_steps=4,
        bf16=True, gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        optim="adamw_torch_fused", lr_scheduler_type="cosine", warmup_ratio=0.05,
        logging_steps=10, save_strategy="no", report_to="none", seed=SEED,
    )
    subset = torch.utils.data.Subset(training_data, range(min(n_samples, len(training_data))))
    return Trainer(model=student, args=args, train_dataset=subset,
                   data_collator=collate_teacher_answers)

sft = make_trainer("sft_1024_lr1e5", 1024, 1e-5, 1)
student = sft.model


Для обучения оставлено 3989 из 4096 ответов


In [44]:
sft.train()
student_path = ensure_empty_dir(ARTIFACTS / "student_runs" / "sft_1024_lr1e5")
sft.model.config.use_cache = True
sft.save_model(student_path)
tokenizer.save_pretrained(student_path)
sft.save_state()
del sft, student
clean_memory()
sft_metrics, sft_predictions = evaluate_checkpoint(student_path, dev_data)
save_evaluation("sft_1024_lr1e5", sft_metrics, sft_predictions, dev_data)
sft_results = [dict(run="sft_1024_lr1e5", n_samples=min(1024, len(training_data)), learning_rate=1e-5,
                    epochs=1, path=str(student_path), **sft_metrics)]
display(pd.DataFrame(sft_results))


Step,Training Loss
10,0.468100
20,0.233000
30,0.210300
40,0.184500
50,0.191400
60,0.198600


  0%|          | 0/64 [00:00<?, ?it/s]

  2%|▏         | 1/64 [00:01<02:03,  1.95s/it]

  3%|▎         | 2/64 [00:03<01:52,  1.81s/it]

  5%|▍         | 3/64 [00:05<01:47,  1.76s/it]

  6%|▋         | 4/64 [00:07<02:01,  2.03s/it]

... (58 outputs left out)

 98%|█████████▊| 63/64 [02:14<00:02,  2.43s/it]

100%|██████████| 64/64 [02:15<00:00,  2.20s/it]

100%|██████████| 64/64 [02:15<00:00,  2.12s/it]

,run,n_samples,learning_rate,epochs,path,quality,intent_accuracy,slot_micro_f1,frame_exact_match,valid_json_rate,n
0,sft_1024_lr1e5,1024,0.00001,1,artifacts_compression_hw/student_runs/sft_1024...,0.531743,0.721102,0.342385,0.252336,0.964092,2033


### Добейтесь максимального качества ученика. Цель — не более 10% отклонения от учителя (2 балла)
Пробуйте менять гиперпараметры, число сэмплов, модели ученика (при условии размера модели ученика не более 8B).

In [45]:
sft_configs = [
    dict(run="sft_1024_lr2e5", n_samples=1024, learning_rate=2e-5, epochs=1),
    dict(run="sft_4096_lr2e5", n_samples=4096, learning_rate=2e-5, epochs=1),
    dict(run="sft_4096_lr2e5_ep2", n_samples=4096, learning_rate=2e-5, epochs=2),
]
for config in sft_configs:
    sft = make_trainer(config["run"], config["n_samples"], config["learning_rate"], config["epochs"])
    sft.train()
    path = ensure_empty_dir(ARTIFACTS / "student_runs" / config["run"])
    sft.model.config.use_cache = True
    sft.save_model(path)
    tokenizer.save_pretrained(path)
    sft.save_state()
    del sft
    clean_memory()
    metrics, predictions = evaluate_checkpoint(path, dev_data)
    save_evaluation(config["run"], metrics, predictions, dev_data)
    sft_results.append({**config, "n_samples": min(config["n_samples"], len(training_data)),
                        "path": str(path), **metrics})
    pd.DataFrame(sft_results).to_csv(ARTIFACTS / "distillation_results.csv", index=False)
    display(pd.DataFrame([sft_results[-1]]))

best = max(sft_results, key=lambda row: row["quality"])
student_path = ARTIFACTS / "student_sft"
shutil.copytree(best["path"], student_path)
for suffix in ("json", "jsonl"):
    shutil.copyfile(ARTIFACTS / "evaluations" / f"{best['run']}.{suffix}",
                    ARTIFACTS / f"student_dev.{suffix}")
comparison = pd.DataFrame([
    {"run": "baseline", **baseline_metrics},
    {"run": "teacher", **teacher_metrics},
    {"run": "student_sft", **{k: best[k] for k in baseline_metrics}},
])
comparison.to_csv(ARTIFACTS / "comparison.csv", index=False)
display(pd.DataFrame(sft_results).drop(columns="path"))
display(comparison)
print("Доля качества учителя:", best["quality"] / teacher_metrics["quality"])
print("Прирост над бейзлайном:", best["quality"] - baseline_metrics["quality"])


Step,Training Loss
10,0.425400
20,0.192700
30,0.204700
40,0.179300
50,0.173500
60,0.171100


  0%|          | 0/64 [00:00<?, ?it/s]

  2%|▏         | 1/64 [00:01<02:03,  1.96s/it]

  3%|▎         | 2/64 [00:03<01:52,  1.82s/it]

  5%|▍         | 3/64 [00:05<01:58,  1.93s/it]

  6%|▋         | 4/64 [00:08<02:07,  2.13s/it]

... (197 outputs left out)

,run,n_samples,learning_rate,epochs,path,quality,intent_accuracy,slot_micro_f1,frame_exact_match,valid_json_rate,n
0,sft_4096_lr2e5_ep2,3989,0.00002,2,artifacts_compression_hw/student_runs/sft_4096...,0.582976,0.774717,0.391234,0.284309,0.997541,2033


,run,n_samples,learning_rate,epochs,quality,intent_accuracy,slot_micro_f1,frame_exact_match,valid_json_rate,n
0,sft_1024_lr1e5,1024,0.00001,1,0.531743,0.721102,0.342385,0.252336,0.964092,2033
1,sft_1024_lr2e5,1024,0.00002,1,0.570575,0.762912,0.378238,0.288736,0.989670,2033
2,sft_4096_lr2e5,3989,0.00002,1,0.568029,0.769306,0.366752,0.269552,0.997049,2033
3,sft_4096_lr2e5_ep2,3989,0.00002,2,0.582976,0.774717,0.391234,0.284309,0.997541,2033


,run,quality,intent_accuracy,slot_micro_f1,frame_exact_match,valid_json_rate,n
0,baseline,0.289967,0.366454,0.213480,0.095917,0.525824,2033
1,teacher,0.561089,0.757009,0.365169,0.281358,0.972946,2033
2,student_sft,0.582976,0.774717,0.391234,0.284309,0.997541,2033


Доля качества учителя: 1.039007561032832
Прирост над бейзлайном: 0.2930090355335537


### Выводы

Quality бейзлайна — 0.2900, учителя — 0.5611, лучшего ученика — 0.5830. Ученик сохраняет 103.90% качества учителя и отличается от бейзлайна на +0.2930, тем самым, победа, цель достигнута. Ученик превосходит бейзлайн. Разница Quality учителя и бейзлайна составляет +0.2711.

Лучшая конфигурация: 3989 ответов учителя, learning rate=2e-05, число эпох — 2. Я сравнил learning rate при одинаковых 1024 примерах, объём обучающих данных при одинаковом learning rate и число эпох на большей выборке. Изменения Quality составили +0.0388, -0.0025 и +0.0149 соответственно. Каждый запуск начинался с исходного бейзлайна.
